# Đối chứng NGÃ / VẬT bằng chuỗi pose — không thay ĐÁNH / CHẠY
ST-GCN chính thức NTU60 → hai nhánh độc lập: ngã và tương tác cặp người.
Hai cấu hình: đóng băng backbone, rồi fine-tune hai tầng cuối với learning rate nhỏ.
Không đổi ôm thành vật; không đổi nằm thành ngã. Không upload video iPhone.
Chỉ dùng 37 video public đã duyệt, nhãn AI hỗ trợ, chưa phải tập nghiệm thu độc lập.
Đây là kiểm tra kiến trúc mới, không phải mở rộng dataset hay bảo đảm cải thiện.
Ngưỡng/checkpoint chọn bằng development; test chỉ báo cáo. Không tự cài lên web.
Cần runtime GPU. Trích pose + hai cấu hình tối đa 30 phút; cài môi trường chạy trước.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, shutil, subprocess, sys, time, urllib.request, zipfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/SENTINEL')
DATA = DRIVE/'datasets'
WORK = Path('/content/three_actions_v1')
CODE = WORK/'code'; DEMO = CODE/'mmaction2_demo'
DEMO.mkdir(parents=True,exist_ok=True)
ARTIFACTS = DRIVE/'artifacts/pose_specialists_v1'
ARTIFACTS.mkdir(parents=True,exist_ok=True)
RUN = ARTIFACTS/('run_'+time.strftime('%Y%m%d_%H%M%S'))
BUNDLE = DRIVE/'artifacts'/('pose_specialists_source_60f7b852f692.zip')
assert BUNDLE.is_file(), 'Thiếu source bundle đã chuẩn bị trên Drive'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == '60f7b852f692ae5df5d28cfbff66b1ee3467161811195be735ad9512ba6f001e', 'Sai source bundle'
with zipfile.ZipFile(BUNDLE) as package:
    assert set(package.namelist()) == {'pose_specialists.py','train_pose_specialists.py','train_three_actions.py','yolov8n-pose.pt','review.json'}
    package.extractall(DEMO)
MANIFEST=DEMO/'review.json'
reviewed=json.loads(MANIFEST.read_text(encoding='utf-8'))
print('37 public clips, giữ nguyên split:',len(reviewed['sources']))
print('Output mới:',RUN)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
def public_source(item):
    target=(DATA/item['video']).resolve()
    assert target.is_relative_to(DATA.resolve())
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest()==item['sha256']:
        return
    url=item['download_url']
    assert url.startswith('https://raw.githubusercontent.com/')
    with urllib.request.urlopen(url,timeout=120) as response:
        content=response.read(item['size']+1)
    assert len(content)==item['size'] and hashlib.sha256(content).hexdigest()==item['sha256']
    temporary=target.with_suffix('.partial'); temporary.write_bytes(content); temporary.replace(target)
with ThreadPoolExecutor(max_workers=3) as pool:
    list(pool.map(public_source,reviewed['sources']))
print('Public sources verified')


In [ ]:
# Môi trường giống pipeline MMAction2 đã dùng để train checkpoint cha.
os.environ.update(MPLBACKEND='Agg', CUDA_MODULE_LOADING='LAZY', OMP_NUM_THREADS='4')
ENV = WORK/'env'; PY = ENV/'bin/python'
def run(command, label, timeout=1800):
    log = ARTIFACTS/('setup_'+label+'.log')
    with log.open('a', encoding='utf-8') as output:
        result = subprocess.run([str(x) for x in command], stdout=output,
                                stderr=subprocess.STDOUT, timeout=timeout)
    if result.returncode:
        raise RuntimeError(f'{label} failed; see {log}\n{log.read_text(encoding="utf-8")[-4000:]}')
    print(label, 'OK')
if not PY.is_file():
    run([sys.executable,'-m','pip','install','-q','uv==0.8.22'], 'install_uv')
    run(['uv','python','install','3.10'], 'install_python')
    run(['uv','venv','--python','3.10','--seed',ENV], 'create_env')
run([PY,'-m','pip','install','-q','numpy==1.26.4','setuptools==69.5.1','wheel'], 'install_base')
run([PY,'-m','pip','install','-q','torch==2.0.1+cu118','torchvision==0.15.2+cu118',
     '--index-url','https://download.pytorch.org/whl/cu118'], 'install_torch')
run([PY,'-m','pip','install','-q','mmcv==2.1.0','--only-binary=mmcv',
     '-f','https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html',
     'mmengine==0.10.7','mmdet==3.2.0','decord==0.6.0','einops==0.8.1',
     'importlib-metadata','scikit-learn==1.3.2','opencv-python==4.8.1.78'], 'install_openmmlab')
if not (DEMO/'repo/.git').is_dir():
    run(['git','clone','--depth','1','--branch','v1.2.0',
         'https://github.com/open-mmlab/mmaction2.git',DEMO/'repo'], 'clone_mmaction')
run([PY,'-m','pip','install','--no-deps','--no-build-isolation','-e',DEMO/'repo'], 'install_mmaction')
os.environ['PYTHONPATH'] = os.pathsep.join(map(str,(CODE,DEMO,DEMO/'repo')))
run([PY,'-c','import torch, mmcv; from mmcv.ops import roi_align; assert torch.cuda.is_available()'], 'check_cuda')

run([PY,'-m','pip','install','--no-deps','ultralytics==8.2.103','ultralytics-thop==2.0.18','psutil==6.1.1','py-cpuinfo==9.0.0'], 'install_pose')


In [ ]:
os.environ['MPLBACKEND']='Agg'
print('Bắt đầu trích pose thật 15 FPS + fine-tune, không dùng nội suy từ 9 frame AVA.')
command=[PY,DEMO/'train_pose_specialists.py','--manifest',MANIFEST,'--root',DATA,
    '--pose-weights',DEMO/'yolov8n-pose.pt','--output',RUN,'--minutes','30','--epochs','80']
log=ARTIFACTS/('train_'+RUN.name+'.log')
with log.open('w',encoding='utf-8') as output:
    process=subprocess.Popen([str(value) for value in command],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,bufsize=1,env=os.environ.copy())
    import threading
    timer=threading.Timer(1830,process.kill); timer.start()
    try:
        for line in process.stdout:
            print(line,end=''); output.write(line); output.flush()
        returncode=process.wait()
    finally:
        timer.cancel()
assert returncode==0, f'Dừng, không tự retry. Xem log: {log}'
report=json.loads((RUN/'comparison.json').read_text())
assert report['status']=='completed'
print(json.dumps(report,ensure_ascii=False,indent=2))
package=shutil.make_archive(str(RUN)+'_pilot','zip',RUN)
print('Gói đối chứng, chưa thay model web:',package)
